In [1]:
using DifferentialEquations
using StaticArrays

In [2]:
R1 = 220
R2 = 220
R3 = 2200
R4 = 22000
R5 = 22000
R6 = 3300
BAT = 15
R7 = 100
R8 = 1000
R9 = 1000
R10 = 1800
C = 100e-9
L = R7*R9*C*R10/R8
C1 = 10e-9
C2 = 100e-9
R = 1800

function chua_params(p)
	(R1,R2,R3,R4,R5,R6,BAT) = p[1:7]
	E1 = R3/(R2+R3)*BAT
	E2 = R6/(R5+R6)*BAT
	m12 = -1/R6
	m02 = 1/R4
	m01 = 1/R1
	m11 = -1/R3
	m1 = m12 + m11
	m0 = m11 + m02
	mm1 = m01 + m02
	Emax = max(E1,E2)
	Emin = min(E1, E2)
	return (E1,E2,m12,m02,m01,m11,m1,m0,mm1,Emax,Emin)
end

function g(x::Number,p::Any)
	
	(E1,E2,m12,m02,m01,m11,m1,m0,mm1,Emax,Emin) = chua_params(p)
	
	if abs.(x) .< Emin
		g = x * m1 
	elseif abs.(x) .< Emax
		g = x*m0
		if x>0
			g = g+ Emin*(m1-m0)
		else
			g = g+Emin*(m0-m1)
		end
	elseif abs.(x) >= Emax
		g = x*mm1
		if x>0 
			g = g+Emax*(m0-mm1) + Emin*(m1-m0)
		else
			g = g+Emax*(mm1-m0) + Emin*(m0-m1)
		end
	end
	return g
end

function g(x::AbstractArray,p::Any)
# function g(x::CuArray,p::Any)


	(E1,E2,m12,m02,m01,m11,m1,m0,mm1,Emax,Emin) = chua_params(p)
	
	mask_low = abs.(x) .< Emin
	mask_mid = (abs.(x) .>= Emin) .& (abs.(x) .< Emax)
	mask_high = abs.(x) .>= Emax

	
	result = similar(x, Float64)

	#Primera zona

	result[mask_low] = x[mask_low] * m1

	# Segunda zona:
	result[mask_mid] = x[mask_mid] * m0 
	
	pos_mask = x .> 0 
	mid_pos = mask_mid .& pos_mask
	mid_neg = mask_mid .& .!pos_mask
	result[mid_pos] .+= Emin*(m1-m0)
	result[mid_neg] .+= Emin*(m0-m1)

	#Tercera zona:
	result[mask_high] = x[mask_high] * mm1

	high_pos = mask_high .& pos_mask 
	high_neg = mask_high .& .!pos_mask 
	result[high_pos] .+= Emax * (m0 - mm1) + Emin * (m1 - m0)
	result[high_neg] .+= Emax * (mm1 - m0) + Emin * (m0 - m1)

	return result
end

g (generic function with 2 methods)

In [3]:
function chua_eq!(du,u,p,t)
	
	(L,C1,C2,R) = p[end-3:end]

	du[1] = (1/C1)*(1/R * (u[2]-u[1]) - g(u[1],p) )

	du[2] = (1/C2)*(1/R * (u[1]-u[2]) + u[3] )

	du[3] = -(1/L)*u[2]
    nothing
end

chua_eq! (generic function with 1 method)

In [4]:
#Rs = 1:0.1:2100;-> Tirada1
#Rs = 1450:0.05:1950 -> Tirada2 
#Rs = 1850:0.01:1940 -> Tirada3
#Rs = 1840:0.1:1940 -> Tirada4
Rs = 1840:0.1:1940

u0 = [-.5; -0.2 ; 0.0];

tspan = (0.00, 0.06);

In [5]:
#Para el multiprocesamiento necesito iterar sobre solo una variable 
    
value_to_index = Dict(Rs[i] => i for i in 1:length(Rs))


prob = ODEProblem(chua_eq!,u0,tspan,zeros(16))
dt = 5e-7
#Tirada 4: 1e-7
#Tirada 3 : 1e-6
#tirada 2  : 1e-6
#tirada 1 : 2.5e-6

#tirada 4 : 1e-6


# t_steady = 0.03:dt:tspan[2]
#Tirada 3 : t_steady = 0.04:dt:tspan[2]
#tirada 4 : t_steady = 0.055:dt:tspan[2]
t_steady = 0.055:dt:tspan[2]

xs=Matrix{Float32}(undef,(length(Rs),length(t_steady)))
ys=Matrix{Float32}(undef,(length(Rs),length(t_steady)))

function run_simulation(Rs,value_to_index,prob)
    time_taken_whole_thing = @elapsed begin
        for R in Rs
                n = value_to_index[R]
            time_taken = @elapsed begin
                
                p0 = (R1,R2,R3,R4,R5,R6,BAT,R7,R8,R9,R10,C,L,C1,C2,R)
                
                # sol = solve_chua_R(p0,tspan,u0)
                prob_R = remake(prob; p=p0) 
        
                # sol = solve(prob_R,alg=TRBDF2(),alg_hints=[:stiff],maxiters=1e10,abstol=1e-10,reltol=1e-10)
                sol = solve(prob_R,alg=ImplicitEuler(),maxiters=1e10,adaptive=false,dt=dt)
                    #Original tol = 1e-10
                sol_array = Array(sol)
            
                stationary_mask = sol.t .> 0.055
                t_steady = sol.t[stationary_mask]
             
                sol_steady = sol_array[:,stationary_mask]
            
                xs[n,:] .= sol_steady[1,:]
                ys[n,:] .= sol_steady[2,:]
                
            end
            if n % 100 == 0
                println("Iteration $n took $time_taken")
            end
            
        end
    end
    println("The whole thing took $time_taken_whole_thing")
end

run_simulation (generic function with 1 method)

In [6]:
run_simulation(Rs,value_to_index,prob)

Iteration 100 took 0.1364061
Iteration 200 took 0.1329989
Iteration 300 took 0.1465067
Iteration 400 took 0.1678558
Iteration 500 took 0.1434565
Iteration 600 took 0.2026837
Iteration 700 took 0.1724895
Iteration 800 took 0.1746037
Iteration 900 took 0.1419603
Iteration 1000 took 0.1448076
The whole thing took 164.5824765


In [7]:
using JLD2

In [8]:
@save "xs.jld2" xs

In [9]:
@save "ys.jld2" ys